# Exercise 8: Hyperpameter tuning/optimization: Random and Grid Search

Pima Indian Diabetes dataset is originally from the National Institute of Diabetes and Digestive and Kidney Diseases. The objective of the dataset is to diagnostically predict whether or not a patient has diabetes, based on certain diagnostic measurements included in the dataset. Several constraints were placed on the selection of these instances from a larger database. In particular, all patients here are females at least 21 years old of Pima Indian heritage.

The datasets consists of several medical predictor variables and one target variable, _Outcome_. Predictor variables includes the number of pregnancies the patient has had, their BMI, insulin level, age, and so on.

[Pima Indian Diabes dataset in Kaggle](https://www.kaggle.com/uciml/pima-indians-diabetes-database)

Dataset can found from Moodle -> Dataset.

__This exercises HAVE TO be done in CSC Puhti cloud environment.__

__NOTE: Try to get as good as possible accuracy. Accurary is one of the evalution criteria__.

TOP results will be found from [here](https://student.labranet.jamk.fi/~rmika/DLNN/TOP-scores.htm)



## Grid Search

Lots of code examples can be found from [this tutorial](https://machinelearningmastery.com/grid-search-hyperparameters-deep-learning-models-python-keras/) for different hyperparameters.

### Use scikit-learn to grid search the batch size, epochs, dropout_rate and learning_rate

In [1]:
# Dowloading reqiuired libraries
%pip install tensorflow
%pip install scikeras

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import libraries
# If needed, run in a separate cell in your Puhti notebook environment:
# %pip install "tensorflow>=2.16,<2.18" "scikeras==0.13.0" "scikit-learn==1.5.2" pandas
# Restart the kernel after installation, then run all cells from the top.
from pathlib import Path
from time import perf_counter
import os

# Limit TensorFlow to the allocated SLURM CPUs on Roihu
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", os.environ.get("SLURM_CPUS_PER_TASK", "1"))
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")

#os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
#os.environ["TF_NUM_INTEROP_THREADS"] = "1"

# Optional: Suppress the TF INFO/WARNING messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from scikeras.wrappers import KerasClassifier
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from IPython.display import Markdown, display

SEED = 42
SEARCH_JOBS = 1
keras.utils.set_random_seed(SEED)



I0000 00:00:1792646974.092577 2728191 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792646974.108915 2728191 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792646979.058100 2728191 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792646979.068718 2728191 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
# Dataset splitting
# # Place your local CSV beside this notebook, or change this path.
# DATA_PATH = Path("pima-indians-diabetes.csv")
# if not DATA_PATH.is_file():
#     raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}. Set DATA_PATH to your local CSV.")

# # Accept both the tutorial's headerless CSV and diabetes.csv with column names.
# data = pd.read_csv(DATA_PATH, header=None)
# first_row = pd.to_numeric(data.iloc[0], errors="coerce")
# if first_row.isna().all():
#     data = data.iloc[1:].reset_index(drop=True)
# data = data.apply(pd.to_numeric, errors="raise")
# if data.shape[1] != 9:
#     raise ValueError("Expected 9 columns: 8 features followed by the binary outcome.")
# if data.iloc[:, -1].isna().any() or not data.iloc[:, -1].isin([0, 1]).all():
#     raise ValueError("The last column must contain binary labels 0 and 1.")

# Load the dataset
data = pd.read_csv("pima-indians-diabetes.csv", header=None)
print(data.info())

# Extract features and labels
features = data.iloc[:, :8].astype(float).copy()
labels = data.iloc[:, 8]

# Handle missing values and later impute them using SimpleImputer in the pipeline
# Zero denotes missing measurements in five columns (Glucose, Blood Pressure, Skin Thickness, Insulin, BMI)
# Pregnancy count and Outcomes column retain their zero values
# Age and Diabetes Pedigree Function columns does not have zero values
features.iloc[:, [1, 2, 3, 4, 5]] = features.iloc[:, [1, 2, 3, 4, 5]].replace(0, np.nan)

# X and Y are only the training data; the final test set stays outside both searches
X, X_test, Y, Y_test = train_test_split(
    features, labels, test_size=0.20,
    stratify=labels, random_state=SEED
)

# Define the cross-validation strategy
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

# Define the grid search parameters as follows:
# learning_rate:[0.001, 0.01, 0.1]
# batch_size: [10, 40 , 80]
# number of epochs: [10, 50, 100]
# dropout_rate:  [0.0, 0.3, 0.5]

# Define the hyperparameter grid for the search as asked in the exercise instructions
learning_rate = [0.001, 0.01, 0.1]
batch_size = [10, 40, 80]
epochs = [10, 50, 100]
dropout_rate = [0.0, 0.3, 0.5]

# NOTE: If you don't have enough memory in your computer, drops some parameters

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       768 non-null    int64  
 1   1       768 non-null    int64  
 2   2       768 non-null    int64  
 3   3       768 non-null    int64  
 4   4       768 non-null    int64  
 5   5       768 non-null    float64
 6   6       768 non-null    float64
 7   7       768 non-null    int64  
 8   8       768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None


In [4]:
#Create model
def create_model(learning_rate=0.001, dropout_rate=0.0):
    keras.backend.clear_session()
    
    #Input layer = 8
    inputs = keras.Input(shape=(8,))
    
    #Only one hidden layer = 12 Dense nodes, activation function = relu
    hidden = keras.layers.Dense(12, activation="relu")(inputs)
    
    #Dropout layer
    hidden = keras.layers.Dropout(dropout_rate)(hidden)
    
    #Output layer: 1 node, activation = sigmoid
    outputs = keras.layers.Dense(1, activation="sigmoid")(hidden)
    
    network = keras.Model(inputs=inputs, outputs=outputs)
    network.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy", metrics=["accuracy"]
    )
    return network


#Define KerasClassifier
classifier = KerasClassifier(
    model=create_model, model__learning_rate=0.001, model__dropout_rate=0.0,
    random_state=SEED, verbose=0
)
# Each CV fold learns its own medians, means, and standard deviations.
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("nn", classifier)
])


In [5]:
#Set parameter dictionary
param_grid = dict(
    nn__model__learning_rate=learning_rate,
    nn__batch_size=batch_size,
    nn__epochs=epochs,
    nn__model__dropout_rate=dropout_rate
)
#Define grid = GridSearcCV(....)
grid = GridSearchCV(
    estimator=model, param_grid=param_grid, n_jobs=SEARCH_JOBS, cv=cv,
    scoring="accuracy", verbose=1, refit=True, error_score="raise"
)
print(f"Grid search: {len(learning_rate) * len(batch_size) * len(epochs) * len(dropout_rate)} candidates, 3 folds each.")


Grid search: 81 candidates, 3 folds each.


In [6]:
#Results
grid_start = perf_counter()
grid_result = grid.fit(X, Y)
grid_seconds = perf_counter() - grid_start


Fitting 3 folds for each of 81 candidates, totalling 243 fits


E0000 00:00:1792646993.041303 2728191 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
E0000 00:00:1792646993.247961 2728191 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
E0000 00:00:1792646998.638261 2728191 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset

E0000 00:00:1792647275.083968 2728191 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
E0000 00:00:1792647280.241641 2728191 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),

In [7]:
#Summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))


Best: 0.786641 using {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.5, 'nn__model__learning_rate': 0.01}
0.767097 (0.014042) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.0, 'nn__model__learning_rate': 0.001}
0.771983 (0.021970) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.0, 'nn__model__learning_rate': 0.01}
0.765487 (0.013545) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.0, 'nn__model__learning_rate': 0.1}
0.760553 (0.014935) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.3, 'nn__model__learning_rate': 0.001}
0.776893 (0.018159) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.3, 'nn__model__learning_rate': 0.01}
0.750821 (0.018154) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__model__dropout_rate': 0.3, 'nn__model__learning_rate': 0.1}
0.747537 (0.008883) with: {'nn__batch_size': 10, 'nn__epochs': 10, 'nn__mo

## Random Search

In [8]:
# Import libraries
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone

#Create model
#Same as in Grid search example 
# Reuse create_model through a fresh clone of the same preprocessing/model pipeline.

#Define KerasClassifier
random_model = clone(model)


In [9]:
# define the Random search parameters
# randomly from same range as the Grid search

#Set parameter dictionary
param_distributions = param_grid.copy()
N_ITER = 20

#Define random_search = RandomizedSearchCV(...)
random_search = RandomizedSearchCV(
    estimator=random_model, param_distributions=param_distributions,
    n_iter=N_ITER, scoring="accuracy", n_jobs=SEARCH_JOBS, cv=cv,
    random_state=SEED, verbose=1, refit=True, error_score="raise"
)
print(f"Random search: {N_ITER} sampled candidates, 3 folds each.")


Random search: 20 sampled candidates, 3 folds each.


In [10]:
#Results
random_start = perf_counter()
random_search.fit(X, Y)
random_seconds = perf_counter() - random_start


Fitting 3 folds for each of 20 candidates, totalling 60 fits


E0000 00:00:1792648752.871679 2728191 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
E0000 00:00:1792648758.967166 2728191 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),

In [11]:
#Summarize results
print("Best: %f using %s" % (random_search.best_score_, random_search.best_params_))
means = random_search.cv_results_['mean_test_score']
stds = random_search.cv_results_['std_test_score']
params = random_search.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))



Best: 0.783365 using {'nn__model__learning_rate': 0.01, 'nn__model__dropout_rate': 0.0, 'nn__epochs': 10, 'nn__batch_size': 40}
0.719863 (0.002953) with: {'nn__model__learning_rate': 0.001, 'nn__model__dropout_rate': 0.3, 'nn__epochs': 10, 'nn__batch_size': 40}
0.767097 (0.014042) with: {'nn__model__learning_rate': 0.001, 'nn__model__dropout_rate': 0.0, 'nn__epochs': 10, 'nn__batch_size': 10}
0.742667 (0.018008) with: {'nn__model__learning_rate': 0.01, 'nn__model__dropout_rate': 0.3, 'nn__epochs': 100, 'nn__batch_size': 10}
0.775251 (0.022103) with: {'nn__model__learning_rate': 0.01, 'nn__model__dropout_rate': 0.3, 'nn__epochs': 10, 'nn__batch_size': 40}
0.776869 (0.018001) with: {'nn__model__learning_rate': 0.001, 'nn__model__dropout_rate': 0.0, 'nn__epochs': 100, 'nn__batch_size': 10}
0.783365 (0.015408) with: {'nn__model__learning_rate': 0.01, 'nn__model__dropout_rate': 0.0, 'nn__epochs': 10, 'nn__batch_size': 40}
0.754065 (0.028028) with: {'nn__model__learning_rate': 0.01, 'nn__mod

# Conclusion

__NOTE: Conclusion is one evaluation criterion for the exercise, so pay attention to writing this__

Write conclusion:


